# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muradlodhi/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames the research question, decision context, and empirical baseline numbers for **Lane 2: Refresh / Content Opportunity Scoring** using the FlyRank starter dataset.

## 1. My lane (or freestyle) and why

**Selected Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**Why this lane:**
Content decay is a high-stakes operational bottleneck for organic search and publishing teams managing large content inventories. In an inventory of tens of thousands of URLs, human editorial bandwidth is strictly constrained: content teams cannot manually review or rewrite every page each quarter. Without calibrated prioritization, editors risk spending valuable hours rewriting evergreen, healthy, or low-demand pages while decaying high-value assets quietly lose Page-1 visibility and organic revenue. Lane 2 directly tackles this operational problem by building an evidence-backed, ranked review queue that pinpoints high-exposure content showing observable decline signals, maximizing the business return on human editorial labor.

In [1]:
# Verify Lane Selection & Environment Setup
from pathlib import Path
import pandas as pd
import numpy as np

print("Selected Lane: Lane 2 (Refresh / Content Opportunity Scoring)")
print("Framework: Decision-support priority queue with transparent reason codes")

Selected Lane: Lane 2 (Refresh / Content Opportunity Scoring)
Framework: Decision-support priority queue with transparent reason codes


## 2. The question: decision, action, cost of a wrong call

- **What decision does this improve?**
  It improves the weekly editorial triage decision: *"Which 25 to 50 existing content URLs should our content team prioritize for a substantive refresh, expansion, or metadata update this sprint?"*

- **Who acts on the output, and what do they do?**
  SEO content editors, content strategists, and subject-matter writers act on the queue. For each flagged URL, they inspect the associated reason codes (such as `page_one_decay_risk`, `stale_visible_page`, or `low_ctr_visible_page`), audit the SERP and on-page content, and execute targeted updates (updating outdated data, improving depth, sharpening title/meta tags, or fixing search intent alignment).

- **What does a wrong recommendation cost?**
  - **False Positives (Recommending stable/healthy content):** Wasted human writing and editing hours spent modifying content that did not need changes, creating an opportunity cost where urgent pages were ignored.
  - **False Negatives (Missing critical decaying content):** High-traffic, revenue-generating pages on Page 1 quietly slip out of top positions, resulting in compounding organic traffic and revenue loss that is harder and slower to regain.

- **Why does data / ML help at all (over a plain rule)?**
  A simple heuristic (e.g., `age > 180 days` or `traffic down > 20%`) creates unmanageable queues flooded with false positives, while ignoring newer decaying pages. Real content decay involves tangled, multi-variable signals — historical impression volume, ranking position tiers, CTR relative to position, engagement velocity, and time since last revision. A learned scoring model balances these interacting trade-offs to produce a calibrated, high-precision ranking that static threshold rules cannot achieve.

In [2]:
# Define Decision Context & Operational Constraints
decision_context = {
    "primary_actor": "SEO Content Strategist / Editorial Lead",
    "decision": "Weekly content refresh queue prioritization",
    "action": "Targeted content revision, metadata update, or intent realignment",
    "cost_of_fp": "Wasted editorial hours on stable content",
    "cost_of_fn": "Unaddressed loss of Page-1 visibility and compounding traffic loss",
    "target_output": "Ranked queue with calibrated opportunity scores and reason codes"
}

for k, v in decision_context.items():
    print(f"{k.replace('_', ' ').title()}: {v}")

Primary Actor: SEO Content Strategist / Editorial Lead
Decision: Weekly content refresh queue prioritization
Action: Targeted content revision, metadata update, or intent realignment
Cost Of Fp: Wasted editorial hours on stable content
Cost Of Fn: Unaddressed loss of Page-1 visibility and compounding traffic loss
Target Output: Ranked queue with calibrated opportunity scores and reason codes


## 3. Quick look at the data (2-3 real numbers)

Let us inspect the starter dataset (`data/raw/content_refresh_anonymized.csv`) to establish key empirical baseline numbers supporting this lane.

In [3]:
# Load and audit starter dataset numbers
data_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

total_rows = len(df)
unique_clients = df["client_id"].nunique()
total_impressions = df["impressions_90d"].sum()

# 1. Base rate of downward trend
down_mask = df["trend_direction"] == "down"
down_count = int(down_mask.sum())
down_pct = (down_count / total_rows) * 100

# 2. High-visibility Page 1 assets currently in decay (avg_position 1-10 & trend_direction == down)
p1_decay_mask = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & down_mask
p1_decay_count = int(p1_decay_mask.sum())
p1_decay_pct = (p1_decay_count / total_rows) * 100

# 3. Impression volume concentration at risk on Page 1
p1_decay_impressions = int(df.loc[p1_decay_mask, "impressions_90d"].sum())
p1_decay_imp_pct = (p1_decay_impressions / total_impressions) * 100

# 4. Stale high-demand assets (days_since_last_update >= 180 and impressions >= 500)
stale_demand_mask = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
stale_demand_count = int(stale_demand_mask.sum())

print(f"=== STARTER DATASET AUDIT ({total_rows:,} Content Items across {unique_clients} Clients) ===")
print(f"1. Overall Decline Base Rate: {down_count:,} / {total_rows:,} items ({down_pct:.2f}%) exhibit a downward trend.")
print(f"2. High-Visibility Page-1 Decay: {p1_decay_count:,} items ({p1_decay_pct:.2f}% of inventory) rank on Page 1 (pos 1-10) yet are in active decline.")
print(f"3. Exposure Concentration: These Page-1 decaying assets represent {p1_decay_impressions:,} impressions ({p1_decay_imp_pct:.2f}% of all inventory demand).")
print(f"4. Stale High-Demand Candidate Pool: {stale_demand_count:,} items have not been refreshed in >= 180 days despite >= 500 impressions.")

=== STARTER DATASET AUDIT (30,000 Content Items across 32 Clients) ===
1. Overall Decline Base Rate: 16,262 / 30,000 items (54.21%) exhibit a downward trend.
2. High-Visibility Page-1 Decay: 7,311 items (24.37% of inventory) rank on Page 1 (pos 1-10) yet are in active decline.
3. Exposure Concentration: These Page-1 decaying assets represent 44,876,369 impressions (28.76% of all inventory demand).
4. Stale High-Demand Candidate Pool: 17 items have not been refreshed in >= 180 days despite >= 500 impressions.


### Key Data Insights:
1. **Decline is Widespread but Unequal:** Over half (**54.21%**, 16,262 items) of all content items exhibit a downward trend over the trailing 90-day window. Editorial teams cannot realistically rewrite 16,000+ pages at once.
2. **High Stakes on Page 1:** **7,311 content items (24.37%)** hold valuable Page-1 positions (positions 1–10) while actively losing momentum. Protecting these assets is critical because they account for **44,876,369 impressions (28.76% of total search demand)** across the 32 clients.
3. **Clear Need for Priority Ranking:** The starter data confirms that a raw threshold or unranked list is overwhelming. An algorithmic ranking model that prioritizes the highest-impact declining pages provides clear operational value.

## 4. Careful words: what I can and can't claim

| What I CAN Claim | What I CANNOT Claim |
|---|---|
| **Observational association:** Observable signals (impression volume, position tiers, age, decay velocity) are empirically correlated with historical traffic movement. | **Causal guarantees:** I cannot claim that updating a page is *guaranteed* to reverse a decline or boost rankings (causality requires randomized controlled experiments). |
| **Decision-support prioritization:** The model produces an ordered review queue with human-interpretable reason codes to optimize human editorial review time. | **Algorithm reverse-engineering:** I am not claiming to model Google's internal search ranking algorithm or predict future search engine updates. |
| **Observable empirical outcomes:** Evaluations measure precision@K, ROC-AUC, and average precision against observed historical outcomes. | **Deterministic traffic forecasting:** I cannot predict exact future click numbers or absolute financial returns from individual edits. |

In [4]:
# Print Guardrail Summary
guardrails = [
    "CLAIM: Observational prioritization and decision-support ranking.",
    "NON-CLAIM: No causal attribution or deterministic rank guarantees.",
    "NON-CLAIM: Not reverse-engineering search engine ranking algorithms.",
    "METHOD: Precision@K and honest out-of-client holdout validation."
]
for g in guardrails:
    print(f"[Guardrail] {g}")

[Guardrail] CLAIM: Observational prioritization and decision-support ranking.
[Guardrail] NON-CLAIM: No causal attribution or deterministic rank guarantees.
[Guardrail] NON-CLAIM: Not reverse-engineering search engine ranking algorithms.
[Guardrail] METHOD: Precision@K and honest out-of-client holdout validation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.